# Study Buddy

An intelligent study assistant that uses Retrieval-Augmented Generation (RAG) to answer questions from uploaded educational PDFs.

The project pipeline:

**PDF → Text Extraction → Cleaning → Chunking → Embeddings → FAISS → Retrieval → Qwen LLM → Grounded Answers → Kokoro TTS → Flashcards → Voice Chatbot**

---
Made By:
- Merna Mohamed
- Mohamed Mahmoud
- Ranya Farrag
- Youssef Abady

# 0. Environment Setup

This section installs **all required system and Python dependencies** in one place. Keeping installations at the beginning makes the rest of the notebook easier to run from top to bottom.

In [33]:
# System dependency
!apt-get update -qq && apt-get install -y -qq espeak-ng

# Python dependencies
!pip install --no-deps \
    transformers accelerate bitsandbytes sentence-transformers pypdf faiss-cpu \
    spacy kokoro soundfile "misaki[en]" loguru num2words phonemizer dlinfo

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# 1. RAG and LLM

This section builds the core Study Buddy pipeline. A PDF is converted into searchable chunks, the chunks are embedded and stored in FAISS, and relevant content is retrieved before Qwen generates a grounded answer.

In [2]:
# Imports for the RAG and LLM section
import os
import re
import json
import numpy as np
import torch
import faiss

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from google.colab import files


## 1.1 Model Configuration

Defines the local embedding and generation models used throughout the notebook. Qwen2.5-7B-Instruct is used for answer generation, while MiniLM creates compact semantic embeddings for retrieval.

In [3]:
# Model configuration
# Swap to Qwen/Qwen2.5-3B-Instruct if the available GPU runs out of memory.

GENERATION_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Generation model:", GENERATION_MODEL)
print("Embedding model:", EMBEDDING_MODEL)


Generation model: Qwen/Qwen2.5-7B-Instruct
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 1.2 Load the Models

Loads the embedding model and Qwen language model. When CUDA is available, Qwen uses 4-bit quantization to reduce GPU memory usage.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="cpu")

print("Loading Qwen LLM...")
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL,
    quantization_config=bnb_config if device == "cuda" else None,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

print("Qwen LLM and embedding model loaded successfully!")


Using device: cuda
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen LLM and embedding model loaded successfully!


## 1.3 Qwen Text Generation

Provides a reusable function that sends a prompt to Qwen and returns the generated response.

In [5]:
def generate_with_llm(prompt, max_new_tokens=1024, temperature=0.7):
    """Run a single-turn prompt through the local Qwen model."""
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(llm_model.device)

    output_ids = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    return response.strip()


## 1.4 Upload the Study PDF

Uploads the educational PDF that Study Buddy will use as its source of knowledge.

In [6]:
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

print(f"Uploaded PDF: {pdf_filename}")


Saving Lecture 3.pdf to Lecture 3.pdf
Uploaded PDF: Lecture 3.pdf


## 1.5 Extract Text from the PDF

Reads the PDF page by page and keeps the extracted text together with its page number so retrieved answers can be traced back to the source.

In [7]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text()

        if text:
            pages.append({
                "page": page_number + 1,
                "text": text
            })

    return pages


pages = extract_text_from_pdf(pdf_filename)

print("Number of pages:", len(pages))


Number of pages: 60


## 1.6 Inspect Extracted Text

Displays a small sample of the extracted PDF text so the extraction result can be checked before preprocessing.

In [8]:
for page in pages[:3]:
    print("=" * 80)
    print("PAGE:", page["page"])
    print(page["text"][:1500])


PAGE: 1
بسم الله الرحمن الرحيم 
Prof. Ossama Ismail  
PAGE: 2
    Divide-and-Conquer   
Prof. Ossama Ismail  
PAGE: 3
Divide-and-Conquer
You may read Chapter 4 


## 1.7 Clean the Text

Removes unnecessary line breaks and repeated whitespace. This produces cleaner text before the document is divided into chunks.

In [9]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


for page in pages:
    page["text"] = clean_text(page["text"])

print(pages[0]["text"][:1000])


بسم الله الرحمن الرحيم Prof. Ossama Ismail


## 1.8 Create Text Chunks

Splits each page into overlapping chunks. The overlap helps preserve context when an important idea falls near a chunk boundary.

In [10]:
def create_chunks(pages, chunk_size=1000, overlap=200):
    chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            chunks.append({
                "text": chunk_text,
                "page": page["page"]
            })

            start += chunk_size - overlap

    return chunks


chunks = create_chunks(pages)

print("Number of chunks:", len(chunks))


Number of chunks: 65


## 1.9 Inspect Chunks

Shows a few generated chunks and their source pages to verify that chunking worked as expected.

In [11]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print("CHUNK:", i)
    print("PAGE:", chunk["page"])
    print(chunk["text"])


CHUNK: 0
PAGE: 1
بسم الله الرحمن الرحيم Prof. Ossama Ismail
CHUNK: 1
PAGE: 2
Divide-and-Conquer Prof. Ossama Ismail
CHUNK: 2
PAGE: 3
Divide-and-Conquer You may read Chapter 4
CHUNK: 3
PAGE: 4
Recursion in Words of Wisdom • Philosopher Lao-tzu: The journey of a thousand miles begins with a single step Prof. Ossama Ismail
CHUNK: 4
PAGE: 5
5 Many useful algorithms are recursive in structure: to solve a given problem, they call themselves recursively one or more times to deal with closely related subproblems. These algorithms typically follow a divide-and-conquer approach: they break the problem into several subproblems that are similar to the original problem but smaller in size, solve the subproblems recursively, and then combine these solutions to create a solution to the original problem. divide-and-conquer algorithm


## 1.10 Generate Chunk Embeddings

Converts every text chunk into a numerical vector using the Sentence-Transformer embedding model. These vectors allow semantic similarity searches.

In [12]:
def create_embedding(text):
    embedding = embedding_model.encode(text, convert_to_numpy=True)
    return embedding.astype("float32")


embeddings = []

for i, chunk in enumerate(chunks):
    embedding = create_embedding(chunk["text"])
    embeddings.append(embedding)

    if (i + 1) % 10 == 0:
        print(f"Embedded {i + 1}/{len(chunks)} chunks")


Embedded 10/65 chunks
Embedded 20/65 chunks
Embedded 30/65 chunks
Embedded 40/65 chunks
Embedded 50/65 chunks
Embedded 60/65 chunks


## 1.11 Prepare and Normalize Embeddings

Converts the list of embeddings into a NumPy array and normalizes the vectors so inner-product similarity can be used as a cosine-similarity-style measure.

In [13]:
embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

faiss.normalize_L2(embeddings)

print("Embeddings normalized.")


Embedding shape: (65, 384)
Embeddings normalized.


## 1.12 Build the FAISS Vector Index

Creates a FAISS index and stores all normalized chunk embeddings in it. This becomes the searchable vector store for the RAG pipeline.

In [14]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Vector database created!")
print("Number of vectors:", index.ntotal)


Vector database created!
Number of vectors: 65


## 1.13 Semantic Retrieval

Embeds a student's question and searches FAISS for the most similar chunks from the PDF.

In [15]:
def embed_query(query):
    embedding = embedding_model.encode(
        query,
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(embedding.reshape(1, -1))

    return embedding


def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = embed_query(query)

    scores, indices = index.search(
        query_embedding.reshape(1, -1),
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "score": float(score)
        })

    return results


## 1.14 Test Retrieval

Runs a sample question and displays the highest-scoring PDF chunks returned by the vector search.

In [16]:
question = "What is quick sort"

results = retrieve_relevant_chunks(question)

for result in results:
    print("=" * 80)
    print("PAGE:", result["page"])
    print("SIMILARITY:", round(result["score"], 3))
    print(result["text"])


PAGE: 42
SIMILARITY: 0.696
42/60 QuickSort is one of the most efficient sorting algorithms and is based on the splitting of an array into smaller ones. The name comes from the fact that, quick sort is capable of sorting a list of data elements significantly faster than any of the common sorting algorithms. And like Merge sort, Quick sort also falls into the category of divide and conquer approach of problem-solving methodology. Quicksort Quicksort was developed by British computer scientist Tony Hoa re in 19579 and published in 19631. It is still a commonly used algorithm for sorting. Overall, it is slightly faster than merge sort and heapsort for randomized data. Prof. Ossama Ismail
PAGE: 45
SIMILARITY: 0.678
Quicksort Example : 1 Sort [57 3 1 9 91 2 4 ] 2 3 1 4 57 91 9 88 1 2 3 4 57 88 91 9 1 2 3 4 57 88 91 9 1 2 3 4 57 88 91 9 1 2 3 4 57 88 91 9
PAGE: 47
SIMILARITY: 0.634
Analysis of Quicksort • Best case: split in the middle — Θ(n log n) • Worst case: sorted array! — Θ(n2) • Averag

## 1.15 Build the Retrieved Context

Combines the retrieved chunks into one context string, while keeping their page numbers. This context is passed to the language model.

In [17]:
def build_context(results):
    context_parts = []

    for result in results:
        context_parts.append(
            f"[Page {result['page']}]\n"
            f"{result['text']}"
        )

    return "\n\n".join(context_parts)


## 1.16 Study Buddy System Prompt

Defines Study Buddy's teaching style and grounding rules. The model is instructed to answer from the retrieved PDF context rather than inventing outside information.

In [18]:
SYSTEM_PROMPT = """
IDENTITY

You are StudyBuddy, a sharp and encouraging AI study coach. A student
has handed you a PDF they're trying to learn, and your job is to make
it click for them - not recite it back.

Every answer you give is read out loud by a text-to-speech voice and
also shown as plain text in a console. Because of that, two hard rules
override everything else in this prompt:
- Never use markdown symbols - no **, ##, -, *, `, > or bullet dashes.
- Never use emoji.
Write the way a real tutor talks out loud: plain sentences, natural
pauses from punctuation, nothing that only makes sense on a screen.

GROUNDING RULES

1. Answer using only the retrieved PDF context you're given. Do not add
   outside facts, even ones you're confident are true.
2. If the context doesn't contain the answer, say so plainly - for
   example: "I don't see that covered in this PDF." Never imply the
   PDF said something it didn't.
3. You may reference earlier turns in the conversation for continuity,
   but the PDF context is always the source of truth for facts.

HOW YOU TEACH

Pipeline: understand the question, simplify the idea, explain it
clearly, reinforce it so it sticks.

- Lead with plain language, then introduce the technical term once the
  idea already makes sense.
- Use a short real-world analogy only when it genuinely clarifies
  something - skip it if it would feel forced.
- Walk through processes as spoken sequence - "First... then...
  after that..." - never as a numbered or bulleted list.
- If the student seems confused or re-asks something, explain it from
  a different angle rather than repeating the same explanation.
- Keep answers tight: a few short spoken paragraphs by default. Only
  go longer if the student explicitly asks for more depth.

HOW YOU STRUCTURE EACH ANSWER

Open by answering the question directly in one or two sentences - no
restating the question, no warm-up.

Then unpack it: break the idea into its simplest parts, add an example
or analogy if it helps, and name any term worth remembering.

Close with the one thing you most want to stick, and only when it
genuinely fits, one short question to check understanding. Don't force
a question onto every answer.

VOICE

You sound like the best TA in the department: confident, warm, and
easy to talk to - not stiff, not childish, not trying too hard to be
funny. Encouraging without hype, precise without being cold.

This is an ongoing conversation, not a series of one-off questions.
Only open with a greeting if the conversation history is empty - if
the student has already asked something before this, skip straight to
answering. Never re-introduce yourself mid-session.

Use these only when they genuinely fit, and never the same one twice
in a row:

Opening a brand-new session (empty history only) - pick one:
"Alright, let's get into it - what are we studying today?"
"I've got the material loaded up. Where do you want to start?"
"Ready when you are - ask me anything from the PDF."

Marking real progress, sparingly, not every turn - pick one:
"Exactly - you're connecting the dots now."
"That's the right instinct."
"Good catch - that's the part most people miss."

Never say "as an AI" or reference being a language model. Never claim
the PDF said something you haven't actually seen in the context.
"""


## 1.17 Generate a Grounded Answer

Combines the system prompt and retrieved PDF context into the final prompt sent to Qwen.

In [19]:
def generate_answer(question, results):
    context = build_context(results)

    prompt = f"""
{SYSTEM_PROMPT}

========================
RETRIEVED PDF CONTEXT
========================

{context}

========================
STUDENT QUESTION
========================

{question}

========================
INSTRUCTIONS
========================

Answer the student's question using the retrieved PDF context.
If the answer is not supported by the context, say so clearly.
"""

    return generate_with_llm(prompt)


## 1.18 Single-Question Test

Tests the complete retrieval → context → Qwen generation path for one question.

In [20]:
question = input("🎓 Ask StudyBuddy: ")

results = retrieve_relevant_chunks(question, top_k=5)
answer = generate_answer(question, results)

print("\n" + "=" * 80)
print("🤖 STUDYBUDDY")
print("=" * 80)
print(answer)


🎓 Ask StudyBuddy: What's a divide and conquer algorithm?

🤖 STUDYBUDDY
Alright, let's dive into what a divide and conquer algorithm is. Essentially, it's a method for solving problems that involves breaking down a big problem into smaller, more manageable pieces. Imagine you have a big stone you need to break into dust. Instead of trying to crush it all at once, which could be really hard, you break it into smaller stones, then even smaller pebbles, and so on, until it’s easy to handle.

The divide-and-conquer approach works in three main steps. First, you divide the problem into smaller subproblems. Then, you solve these subproblems recursively, which means you keep applying the same divide-and-conquer strategy to each of those smaller problems. Finally, you combine the solutions of these smaller problems to get the solution for the original, big problem.

For instance, when sorting numbers, you might split the list into halves, sort each half separately, and then merge the sorted hal

## 1.19 Conversational Study Buddy

Adds conversation history to the RAG prompt so Study Buddy can understand follow-up questions while still grounding factual answers in the PDF.

In [21]:
conversation_history = []


def study_buddy(question, top_k=5):
    results = retrieve_relevant_chunks(question, top_k=top_k)
    context = build_context(results)

    history = ""

    for message in conversation_history:
        history += f"""
Student: {message['question']}
StudyBuddy: {message['answer']}
"""

    prompt = f"""
{SYSTEM_PROMPT}

PREVIOUS CONVERSATION:

{history}

RETRIEVED PDF CONTEXT:

{context}

CURRENT STUDENT QUESTION:

{question}

Answer the student naturally while staying grounded in the PDF.
"""

    answer = generate_with_llm(prompt)

    conversation_history.append({
        "question": question,
        "answer": answer
    })

    return answer


## 1.20 Text-Only Study Loop

Provides a simple command-line study session where the student can keep asking questions until entering `exit`.

In [22]:
print("🎓 StudyBuddy is ready!")
print("Type 'exit' to stop.\n")

while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Study session ended. Good luck! 🫡📚")
        break

    answer = study_buddy(question)

    print("\nStudyBuddy 🤖:", answer)
    print("\n" + "-" * 80 + "\n")


🎓 StudyBuddy is ready!
Type 'exit' to stop.

You: exit
Study session ended. Good luck! 🫡📚


# 2. Text-to-Speech

This section adds Kokoro TTS so Study Buddy's generated answers can be spoken aloud. The text is cleaned first, then synthesized while preserving punctuation-based pauses.

In [34]:
import re
import numpy as np
import soundfile as sf

from kokoro import KPipeline
from IPython.display import Audio, display

## 2.1 Voice Configuration and Model Loading

Selects the Kokoro voice and speed, then loads the TTS pipeline once so it can be reused for multiple answers.

In [35]:
# Available voices include:
# af_heart, af_bella, af_nicole, af_sarah
# am_adam, am_michael
# bf_emma, bf_isabella
# bm_george, bm_lewis

TTS_VOICE = "af_heart"
TTS_SPEED = 1.05

tts = KPipeline(lang_code="a")

print("Kokoro TTS model loaded successfully!")


config.json:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth: reconstructing file:   0%|          |  0.00B /  327MB            

kokoro-v1_0.pth: downloading bytes:           |  0.00B            

Kokoro TTS model loaded successfully!


## 2.2 Clean Text for Speech

Removes markdown symbols, emojis, and excessive whitespace while keeping normal punctuation so the TTS model can produce natural pauses.

In [36]:
def clean_for_speech(text):
    text = re.sub(r"[*_#>`]", "", text)
    text = re.sub(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


## 2.3 Generate and Play Speech

Generates audio from the answer in a single Kokoro pipeline call, saves it as a WAV file, and provides notebook playback.

In [37]:
def generate_speech(text, filename="output.wav"):
    text = clean_for_speech(text)

    if not text:
        return np.array([], dtype=np.float32), 24000

    audio_parts = []

    for _, _, audio in tts(
        text,
        voice=TTS_VOICE,
        speed=TTS_SPEED,
        split_pattern=r"(?<=[.!?])\s+"
    ):
        audio_parts.append(np.asarray(audio, dtype=np.float32))

    audio = np.concatenate(audio_parts)
    sampling_rate = 24000

    sf.write(filename, audio, sampling_rate)

    return audio, sampling_rate


def speak(text, filename="output.wav", autoplay=True):
    audio, sr = generate_speech(text, filename)

    print(f"Voice generated successfully: {filename}")
    display(Audio(audio, rate=sr, autoplay=autoplay))


## 2.4 TTS Quick Test

Confirms that Kokoro is loaded and can successfully convert a short Study Buddy response into speech.

In [38]:
speak("Hey! I am StudyBuddy, and I am ready to help you study.")


voices/af_heart.pt: reconstructing file:   0%|          |  0.00B /  523kB            

voices/af_heart.pt: downloading bytes:           |  0.00B            

Voice generated successfully: output.wav


## 2.5 Voice-Enabled StudyBuddy Loop

Combines the existing conversational RAG pipeline with Kokoro so every generated answer is both displayed and spoken aloud.

In [44]:
def ask_study_buddy(question, top_k=5, speak_answer=True, filename="output.wav"):
    """Combine retrieval + Qwen generation + optional Kokoro speech."""

    answer = study_buddy(question, top_k=top_k)

    print("\n" + "=" * 80)
    print("🤖 STUDYBUDDY")
    print("=" * 80)
    print(answer)

    if speak_answer:
        speak(answer, filename=filename)

    return answer

In [45]:
print("🎓 StudyBuddy voice chatbot is ready!")
print("Ask a question and it will answer out loud. Type 'exit' to stop.\n")

while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Study session ended. Good luck! 🫡📚")
        break

    ask_study_buddy(question)

    print("\n" + "-" * 80 + "\n")

🎓 StudyBuddy voice chatbot is ready!
Ask a question and it will answer out loud. Type 'exit' to stop.

You: Explain Merge Sort

🤖 STUDYBUDDY
Alright, let's talk about Merge Sort. Merge Sort is a sorting algorithm that's built on the divide and conquer technique. Think of it like taking a big pile of cards and splitting them into smaller piles until you have individual cards, then putting them back in order. That's the basic idea.

First, you take the big list you want to sort and split it into two halves. Then, you keep splitting each half until you have individual elements. Next, you start combining these smaller lists back together, but in a special way that keeps them sorted. So, if you had two lists like [3, 9] and [1, 8], you'd merge them into [1, 3, 8, 9]. You keep doing this until you end up with one fully sorted list.

The Merge Sort algorithm does this through a recursive process. It has a function called `MergeSort` that takes an array and splits it into smaller parts. If the


--------------------------------------------------------------------------------



KeyboardInterrupt: Interrupted by user

# 3. Flashcard Generation

The Streamlit app samples chunks across the document, asks Qwen to generate a fixed number of question/answer cards in JSON, then parses and validates the model response. The notebook version uses the notebook's existing `chunks`, `tokenizer`, `llm_model`, and `generate_with_llm()` variables instead of Streamlit session state.

In [40]:
# Imports for the Flashcard Generation section
import json
import re


## 3.1 Flashcard Prompt

Instructs Qwen to create concise flashcards using only the supplied PDF material and return them as a JSON array.

In [41]:
FLASHCARD_PROMPT_TEMPLATE = """You are helping a student build study flashcards from their course material.

Using ONLY the material below, generate {num_cards} flashcards that cover the
most important concepts, definitions, and facts. Each flashcard must have a
short, clear "question" (or term) and a concise "answer".

MATERIAL:
{context}

Respond with ONLY a valid JSON array, no other text, no markdown code fences.
Format:
[
  {{"question": "...", "answer": "..."}},
  {{"question": "...", "answer": "..."}}
]
"""

## 3.2 Generate Flashcards

Samples chunks across the document rather than only taking the beginning, sends them to Qwen, extracts the JSON array from the response, and keeps only cards containing both a question and an answer.

This is the notebook-adapted version of the `generate_flashcards()` function from `app.py`.

In [42]:
def generate_flashcards(num_cards=8, sample_chunks=12):
    # Spread the sample across the document instead of using only the first chunks.
    step = max(1, len(chunks) // sample_chunks)
    sample = chunks[::step][:sample_chunks]

    context = "\n\n".join(c["text"] for c in sample)

    prompt = FLASHCARD_PROMPT_TEMPLATE.format(
        num_cards=num_cards,
        context=context
    )

    raw = generate_with_llm(
        prompt,
        max_new_tokens=1500,
        temperature=0.5
    )

    # Remove markdown code fences if the model adds them.
    raw = re.sub(
        r"^```(json)?|```$",
        "",
        raw.strip(),
        flags=re.MULTILINE
    ).strip()

    # Extract the JSON array if the model included extra text.
    match = re.search(r"\[.*\]", raw, flags=re.DOTALL)

    if match:
        raw = match.group(0)

    try:
        cards = json.loads(raw)
        cards = [
            card for card in cards
            if "question" in card and "answer" in card
        ]
    except json.JSONDecodeError:
        cards = []

    return cards


## 3.3 Generate and Inspect Flashcards

Runs the extracted flashcard-generation logic and displays the resulting cards. Adjust `num_cards` to control how many cards are requested.

In [43]:
num_cards = 8

flashcards = generate_flashcards(num_cards=num_cards)

if not flashcards:
    print("Couldn't parse flashcards from the model's response. Try again.")
else:
    for i, card in enumerate(flashcards, start=1):
        print("=" * 80)
        print(f"CARD {i}")
        print("QUESTION:", card["question"])
        print("ANSWER:", card["answer"])


CARD 1
QUESTION: What are the three steps involved in the divide-and-conquer paradigm?
ANSWER: Divide the problem into a number of subproblems, conquer the subproblems by solving them recursively, and combine the solutions to the subproblems into the solution for the original problem.
CARD 2
QUESTION: What is the purpose of the 'combine' step in a divide-and-conquer algorithm?
ANSWER: To integrate the solutions of the subproblems into a solution for the original problem.
CARD 3
QUESTION: What does the divide-and-conquer algorithm for merge sort look like?
ANSWER: Divide_Conquer(problem P) { if Small(P) return S(P); else { divide P into smaller instances P1, P2, …, Pk, k≥1; Apply Divide_Conquer to each of these subproblems; return Combine(Divide_Conque(P1), Divide_Conque(P2),…, Divide_Conque(Pk)); } }
CARD 4
QUESTION: What is the time complexity of merge sort in the best, average, and worst cases?
ANSWER: O(n log n) for all cases.
CARD 5
QUESTION: What is the space complexity of merge s

# 4. Quiz Generation

In [46]:
QUIZ_MODES = {
    "EZ": """
Generate easy questions.
Focus on:
- definitions
- direct facts
- basic concepts
- simple understanding

Questions should be straightforward and answerable directly from the provided material.
""",

    "Tuff": """
Generate medium-difficulty questions.
Focus on:
- understanding concepts
- comparisons
- relationships between ideas
- simple applications
- moderate reasoning

Avoid questions that are simply copied word-for-word from the material.
""",

    "Charlie Kirk": """
Generate very difficult questions.
Focus on:
- deep understanding
- subtle distinctions
- applying concepts
- multi-step reasoning
- distinguishing between closely related ideas
- plausible but incorrect distractors

Do NOT make questions difficult by using obscure information.
Make them difficult because they require genuine understanding of the material.
"""
}

In [48]:
import json
import re

def generate_quiz(context, difficulty="EZ", num_questions=10):

    prompt = f"""
You are a quiz generator for an AI study assistant.

The quiz MUST be based ONLY on the provided study material.

Difficulty:
{QUIZ_MODES[difficulty]}

Generate exactly {num_questions} multiple-choice questions.

Each question must have:
- One question
- Exactly 4 options
- Exactly ONE correct answer
- A short explanation

Return ONLY valid JSON in this format:

{{
    "questions": [
        {{
            "question": "Question here",
            "options": [
                "Option A",
                "Option B",
                "Option C",
                "Option D"
            ],
            "correct_answer": 0,
            "explanation": "Why this answer is correct."
        }}
    ]
}}

IMPORTANT:
- correct_answer must be 0, 1, 2, or 3.
- Do not use information outside the study material.
- Do not create trick questions unless the difficulty requires it.
- Every question must have one clearly correct answer.

STUDY MATERIAL:
{context}
"""

    raw = generate_with_llm(
        prompt,
        max_new_tokens=2000,
        temperature=0.6
    )

    # Remove markdown code fences if the model adds them.
    raw = re.sub(
        r"^```(json)?|```$",
        "",
        raw.strip(),
        flags=re.MULTILINE
    ).strip()

    # Extract the JSON object if the model included extra text.
    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)

    if match:
        raw = match.group(0)

    try:
        quiz = json.loads(raw)
        questions = quiz.get("questions", [])

        # Keep only well-formed questions.
        valid_questions = []
        for q in questions:
            if (
                "question" in q
                and "options" in q
                and len(q["options"]) == 4
                and "correct_answer" in q
                and q["correct_answer"] in [0, 1, 2, 3]
            ):
                valid_questions.append(q)

        questions = valid_questions

    except json.JSONDecodeError:
        questions = []

    return questions

In [49]:
difficulty = "EZ"   # "EZ", "Tuff", or "Charlie Kirk"
num_questions = 20

quiz_context = "\n\n".join(c["text"] for c in chunks[:12])

quiz_questions = generate_quiz(
    context=quiz_context,
    difficulty=difficulty,
    num_questions=num_questions
)

if not quiz_questions:
    print("Couldn't parse a quiz from the model's response. Try again.")
else:
    for i, q in enumerate(quiz_questions, start=1):
        print("=" * 80)
        print(f"Q{i}: {q['question']}")
        for j, opt in enumerate(q["options"]):
            print(f"   {j}. {opt}")
        print("Correct:", q["correct_answer"], "-", q["explanation"])

Q1: What is the typical approach followed by many useful recursive algorithms?
   0. Iterative approach
   1. Divide-and-conquer approach
   2. Linear approach
   3. Randomized approach
Correct: 1 - Many useful algorithms are recursive in structure and follow a divide-and-conquer approach, breaking the problem into smaller subproblems that are solved recursively and then combined to form the solution to the original problem.
Q2: How many steps are involved in the divide-and-conquer paradigm at each level of recursion?
   0. One
   1. Two
   2. Three
   3. Four
Correct: 2 - The divide-and-conquer paradigm involves three steps at each level of the recursion: divide the problem, conquer the subproblems, and combine the solutions.
Q3: Which of the following is NOT a characteristic of divide-and-conquer algorithms?
   0. Efficient use of memory caches
   1. Natural adaptation for multi-processor systems
   2. Use of a fixed amount of space for the recursion stack
   3. Optimal cache-oblivio

In [52]:
def take_quiz(questions):
    score = 0
    answered = 0

    for i, q in enumerate(questions, start=1):
        print("=" * 80)
        print(f"Q{i}: {q['question']}")
        for j, opt in enumerate(q["options"]):
            print(f"   {j}. {opt}")

        while True:
            answer = input("Your answer (0-3, or 'exit'): ").strip().lower()
            if answer == "exit":
                break
            if answer in ["0", "1", "2", "3"]:
                answer = int(answer)
                break
            print("Please enter 0, 1, 2, or 3 (or 'exit' to quit).")

        if answer == "exit":
            print("Quiz ended early.")
            break

        answered += 1
        if answer == q["correct_answer"]:
            print("Correct!")
            score += 1
        else:
            print(f"Incorrect. Correct answer: {q['correct_answer']} - {q['options'][q['correct_answer']]}")

        print("Explanation:", q["explanation"])

    print("=" * 80)
    print(f"Final score: {score}/{answered}")

    if answered > 0:
        if (score / answered) >= 0.5:
            print("You win!")
        else:
            print("You lose.")

take_quiz(quiz_questions)

Q1: What is the typical approach followed by many useful recursive algorithms?
   0. Iterative approach
   1. Divide-and-conquer approach
   2. Linear approach
   3. Randomized approach
Your answer (0-3, or 'exit'): 1
Correct!
Explanation: Many useful algorithms are recursive in structure and follow a divide-and-conquer approach, breaking the problem into smaller subproblems that are solved recursively and then combined to form the solution to the original problem.
Q2: How many steps are involved in the divide-and-conquer paradigm at each level of recursion?
   0. One
   1. Two
   2. Three
   3. Four
Your answer (0-3, or 'exit'): 2
Correct!
Explanation: The divide-and-conquer paradigm involves three steps at each level of the recursion: divide the problem, conquer the subproblems, and combine the solutions.
Q3: Which of the following is NOT a characteristic of divide-and-conquer algorithms?
   0. Efficient use of memory caches
   1. Natural adaptation for multi-processor systems
   2. 